# AI Inventory Demand & Stock Risk Prediction System
## End-to-End PySpark Pipeline: Ingestion, Feature Engineering & MLlib Modeling

This notebook implements an industry-grade Big Data workflow using **Apache Spark (PySpark)**.
It executes entirely in the JVM and PySpark runtime, avoiding Windows DLL security issues while guaranteeing production scalability.

### 1. Initialize PySpark with Windows & Local Environment Configuration

In [2]:
import os, sys
from pathlib import Path

# Set Hadoop winutils and Python worker paths
project_root = Path.cwd().parent
hadoop_home = project_root / "hadoop"
if hadoop_home.exists():
    os.environ["HADOOP_HOME"] = str(hadoop_home)
    os.environ["PATH"] = str(hadoop_home / "bin") + ";" + os.environ.get("PATH", "")

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Isolate PySpark from broken Windows Pandas C-extensions
sys.modules['pandas'] = None

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("inventory-demand-pyspark-notebook")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("PySpark session initialized! Spark Version:", spark.version)

PySpark session initialized! Spark Version: 4.2.0


### 2. Ingest Raw Dataset & Schema Inspection (76,000 Rows)

In [3]:
csv_path = project_root / "data" / "raw" / "sales_data.csv"
df = spark.read.option("header", True).option("inferSchema", True).csv(str(csv_path))

print("Total Records:", df.count())
df.printSchema()
df.show(5, truncate=False)

Total Records: 76000
root
 |-- Date: date (nullable = true)
 |-- Store ID: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Inventory Level: integer (nullable = true)
 |-- Units Sold: integer (nullable = true)
 |-- Units Ordered: integer (nullable = true)
 |-- Price: double (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Weather Condition: string (nullable = true)
 |-- Promotion: integer (nullable = true)
 |-- Competitor Pricing: double (nullable = true)
 |-- Seasonality: string (nullable = true)
 |-- Epidemic: integer (nullable = true)
 |-- Demand: integer (nullable = true)

+----------+--------+----------+-----------+------+---------------+----------+-------------+-----+--------+-----------------+---------+------------------+-----------+--------+------+
|Date      |Store ID|Product ID|Category   |Region|Inventory Level|Units Sold|Units Ordered|Price|Discount|Weather Co

### 3. Data Quality, Null Values & Cardinality Checks

In [4]:
# Check missing values across all columns
null_counts = df.select([
    F.sum(F.col(f"`{c}`").isNull().cast("int")).alias(c)
    for c in df.columns
]).first().asDict()
print("Missing values per column:", null_counts)

date_stats = df.select(F.min("Date").alias("min_date"), F.max("Date").alias("max_date")).first()
print("Date Range:", date_stats["min_date"], "to", date_stats["max_date"])
print("Unique Stores:", df.select("Store ID").distinct().count())
print("Unique Products:", df.select("Product ID").distinct().count())

Missing values per column: {'Date': 0, 'Store ID': 0, 'Product ID': 0, 'Category': 0, 'Region': 0, 'Inventory Level': 0, 'Units Sold': 0, 'Units Ordered': 0, 'Price': 0, 'Discount': 0, 'Weather Condition': 0, 'Promotion': 0, 'Competitor Pricing': 0, 'Seasonality': 0, 'Epidemic': 0, 'Demand': 0}
Date Range: 2022-01-01 to 2024-01-30
Unique Stores: 5
Unique Products: 20


### 4. Categorical Distributions & Business Metrics Summary

In [5]:
print("Distribution by Category:")
df.groupBy("Category").count().orderBy(F.desc("count")).show()

print("Distribution by Region:")
df.groupBy("Region").count().orderBy(F.desc("count")).show()

print("Summary of Sales and Inventory Metrics:")
df.select("Demand", "Units Sold", "Inventory Level", "Units Ordered", "Price").summary().show()

Distribution by Category:


+-----------+-----+
|   Category|count|
+-----------+-----+
|  Groceries|30400|
|  Furniture|13680|
|   Clothing|12160|
|       Toys|10640|
|Electronics| 9120|
+-----------+-----+

Distribution by Region:
+------+-----+
|Region|count|
+------+-----+
| North|30400|
|  West|15200|
|  East|15200|
| South|15200|
+------+-----+

Summary of Sales and Inventory Metrics:
+-------+------------------+-----------------+------------------+-----------------+-----------------+
|summary|            Demand|       Units Sold|   Inventory Level|    Units Ordered|            Price|
+-------+------------------+-----------------+------------------+-----------------+-----------------+
|  count|             76000|            76000|             76000|            76000|            76000|
|   mean|104.31715789473684|88.82731578947369|301.06284210526314|89.09064473684211|67.72602750000047|
| stddev| 46.96480105784746|43.99452495999812|226.51016094087697|162.4046274705084|39.37789879880463|
|    min|             

### 5. Leakage-Free Feature Engineering (Lags, Rolling Averages & 7-Day Target)

In [6]:
# Window partitioned by store and product, ordered by Date
sp_window = Window.partitionBy("Store ID", "Product ID").orderBy("Date")

# Rolling averages strictly exclude day t (rows -7 to -1) to avoid data leakage
rolling_7 = sp_window.rowsBetween(-7, -1)
rolling_14 = sp_window.rowsBetween(-14, -1)
rolling_28 = sp_window.rowsBetween(-28, -1)

# Target: Cumulative 7-day future demand (rows 1 to 7)
target_7d = sp_window.rowsBetween(1, 7)

featured_df = (
    df
    .withColumn("day_of_week", F.dayofweek("Date"))
    .withColumn("month", F.month("Date"))
    .withColumn("demand_lag_1", F.lag("Demand", 1).over(sp_window))
    .withColumn("demand_lag_7", F.lag("Demand", 7).over(sp_window))
    .withColumn("demand_lag_14", F.lag("Demand", 14).over(sp_window))
    .withColumn("demand_lag_28", F.lag("Demand", 28).over(sp_window))
    .withColumn("rolling_demand_mean_7", F.round(F.avg("Demand").over(rolling_7), 2))
    .withColumn("rolling_demand_mean_14", F.round(F.avg("Demand").over(rolling_14), 2))
    .withColumn("rolling_demand_mean_28", F.round(F.avg("Demand").over(rolling_28), 2))
    .withColumn("target_demand_next_7d", F.sum("Demand").over(target_7d))
    .withColumn("future_days_count", F.count("Demand").over(target_7d))
    .filter(F.col("demand_lag_28").isNotNull() & (F.col("future_days_count") == 7))
    .drop("future_days_count")
)

print("Leakage-free dataset rows ready for modeling:", featured_df.count())
featured_df.select("Date", "Store ID", "Product ID", "Demand", "demand_lag_1", "demand_lag_7", "rolling_demand_mean_7", "target_demand_next_7d").show(5)

Leakage-free dataset rows ready for modeling: 72500
+----------+--------+----------+------+------------+------------+---------------------+---------------------+
|      Date|Store ID|Product ID|Demand|demand_lag_1|demand_lag_7|rolling_demand_mean_7|target_demand_next_7d|
+----------+--------+----------+------+------------+------------+---------------------+---------------------+
|2022-01-29|    S001|     P0001|    81|         194|         102|               110.57|                  642|
|2022-01-30|    S001|     P0001|    79|          81|         126|               107.57|                  624|
|2022-01-31|    S001|     P0001|   113|          79|          44|               100.86|                  576|
|2022-02-01|    S001|     P0001|    90|         113|          61|               110.71|                  621|
|2022-02-02|    S001|     P0001|    94|          90|         160|               114.86|                  647|
+----------+--------+----------+------+------------+------------+---

### 6. Chronological Time-Based Data Split

In [7]:
train_cutoff = "2023-06-15"
val_cutoff = "2023-10-15"

train_df = featured_df.filter(F.col("Date") <= train_cutoff).cache()
val_df = featured_df.filter((F.col("Date") > train_cutoff) & (F.col("Date") <= val_cutoff)).cache()
test_df = featured_df.filter(F.col("Date") > val_cutoff).cache()

print("Training set count:  ", train_df.count())
print("Validation set count:", val_df.count())
print("Test set count:      ", test_df.count())

Training set count:   50300
Validation set count: 12200
Test set count:       10000


### 7. Baseline Models Benchmark (Naive Lag vs. Rolling Average)

In [8]:
from pyspark.ml.evaluation import RegressionEvaluator

def calc_wape(df, label_col="target_demand_next_7d", pred_col="prediction"):
    m = df.select(
        F.sum(F.abs(F.col(label_col) - F.col(pred_col))).alias("abs_err"),
        F.sum(F.col(label_col)).alias("actual")
    ).first()
    return (m["abs_err"] / m["actual"]) * 100.0

mae_eval = RegressionEvaluator(labelCol="target_demand_next_7d", predictionCol="prediction", metricName="mae")

# Baseline 1: Recent Lag 1 x 7
val_b1 = val_df.withColumn("prediction", F.col("demand_lag_1") * 7.0)
print(f"[Baseline 1 - Naive Lag1 x 7]  Validation MAE: {mae_eval.evaluate(val_b1):.2f} | WAPE: {calc_wape(val_b1):.2f}%")

# Baseline 2: 7-day Historical Rolling Average x 7
val_b2 = val_df.withColumn("prediction", F.col("rolling_demand_mean_7") * 7.0)
print(f"[Baseline 2 - Rolling 7d Avg]   Validation MAE: {mae_eval.evaluate(val_b2):.2f} | WAPE: {calc_wape(val_b2):.2f}%")

[Baseline 1 - Naive Lag1 x 7]  Validation MAE: 236.24 | WAPE: 30.56%
[Baseline 2 - Rolling 7d Avg]   Validation MAE: 132.40 | WAPE: 17.13%


### 8. Spark MLlib Machine Learning Pipeline (Random Forest Regressor)

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor

cat_cols = ["Store ID", "Product ID", "Category", "Region", "Weather Condition", "Seasonality"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c.lower().replace(' ', '_')}_idx", handleInvalid="keep") for c in cat_cols]

num_features = [
    "Inventory Level", "Price", "Discount", "Promotion", "Competitor Pricing", "Epidemic",
    "day_of_week", "month", "demand_lag_1", "demand_lag_7", "demand_lag_14", "demand_lag_28",
    "rolling_demand_mean_7", "rolling_demand_mean_14", "rolling_demand_mean_28"
]
all_features = num_features + [f"{c.lower().replace(' ', '_')}_idx" for c in cat_cols]

assembler = VectorAssembler(inputCols=all_features, outputCol="features", handleInvalid="keep")
rf = RandomForestRegressor(featuresCol="features", labelCol="target_demand_next_7d", predictionCol="prediction", numTrees=35, maxDepth=8, seed=42)

pipeline = Pipeline(stages=indexers + [assembler, rf])
print("Fitting Spark MLlib Random Forest model...")
model = pipeline.fit(train_df)

val_preds = model.transform(val_df).cache()
test_preds = model.transform(test_df).cache()

print(f"[Spark ML Random Forest] Validation MAE: {mae_eval.evaluate(val_preds):.2f} | WAPE: {calc_wape(val_preds):.2f}%")
print(f"[Spark ML Random Forest] HELD-OUT TEST MAE: {mae_eval.evaluate(test_preds):.2f} | WAPE: {calc_wape(test_preds):.2f}%")

Fitting Spark MLlib Random Forest model...
[Spark ML Random Forest] Validation MAE: 99.40 | WAPE: 12.86%
[Spark ML Random Forest] HELD-OUT TEST MAE: 105.36 | WAPE: 14.79%


### 9. Inventory Risk Engine & Business Decision Output

In [10]:
# Convert forecast into stock-out risk and holding cost recommendations
risk_df = (
    test_preds
    .withColumn("inventory_gap", (F.col("Inventory Level") + F.col("Units Ordered")) - F.col("prediction"))
    .withColumn("daily_rate", F.when(F.col("prediction") / 7.0 > 0.1, F.col("prediction") / 7.0).otherwise(0.1))
    .withColumn("days_of_cover", F.round(F.col("Inventory Level") / F.col("daily_rate"), 1))
    .withColumn("stock_risk_flag",
        F.when(F.col("inventory_gap") < 0, "HIGH_STOCKOUT_RISK")
        .when(F.col("days_of_cover") > 40, "CRITICAL_OVERSTOCK")
        .otherwise("BALANCED")
    )
    .withColumn("action_recommendation",
        F.when(F.col("stock_risk_flag") == "HIGH_STOCKOUT_RISK", "EXPEDITE_REPLENISHMENT")
        .when(F.col("stock_risk_flag") == "CRITICAL_OVERSTOCK", "REVIEW_PROMOTIONS_PAUSE_PO")
        .otherwise("NORMAL_CYCLE")
    )
)

print("Sample Store-Level Inventory Decision Recommendations:")
risk_df.select("Date", "Store ID", "Product ID", "Inventory Level", "prediction", "inventory_gap", "days_of_cover", "stock_risk_flag", "action_recommendation").show(10, truncate=False)

Sample Store-Level Inventory Decision Recommendations:
+----------+--------+----------+---------------+------------------+-------------------+-------------+------------------+----------------------+
|Date      |Store ID|Product ID|Inventory Level|prediction        |inventory_gap      |days_of_cover|stock_risk_flag   |action_recommendation |
+----------+--------+----------+---------------+------------------+-------------------+-------------+------------------+----------------------+
|2023-10-16|S001    |P0001     |196            |444.2287555224872 |-248.22875552248718|3.1          |HIGH_STOCKOUT_RISK|EXPEDITE_REPLENISHMENT|
|2023-10-17|S001    |P0001     |168            |438.4269332706217 |-150.4269332706217 |2.7          |HIGH_STOCKOUT_RISK|EXPEDITE_REPLENISHMENT|
|2023-10-18|S001    |P0001     |126            |441.274492724231  |-315.274492724231  |2.0          |HIGH_STOCKOUT_RISK|EXPEDITE_REPLENISHMENT|
|2023-10-19|S001    |P0001     |229            |479.2373383424254 |-250.237338342